***

*Course:* [Math 535](https://people.math.wisc.edu/~roch/mmids/) - Mathematical Methods in Data Science (MMiDS)  
*Chapter:* 3-Singular value decomposition   
*Author:* [Sebastien Roch](https://people.math.wisc.edu/~roch/), Department of Mathematics, University of Wisconsin-Madison  
*Updated:* Jan 4, 2024   
*Copyright:* &copy; 2024 Sebastien Roch

***

In [ ]:
# IF RUNNING ON GOOGLE COLAB, UNCOMMENT THE FOLLOWING CODE CELL
# When prompted, upload: 
#     * mmids.py
#     * h3n2-snp.csv
#     * h3n2-other.csv 
#     * advertising.csv 
# from your local file system
# Files at: https://github.com/MMiDS-textbook/MMiDS-textbook.github.io/tree/main/utils
# Alternative instructions: https://colab.research.google.com/notebooks/io.ipynb

In [ ]:
#from google.colab import files
#
#uploaded = files.upload()
#
#for fn in uploaded.keys():
#    print('User uploaded file "{name}" with length {length} bytes'.format(
#      name=fn, length=len(uploaded[fn])))

In [ ]:
# FOR BOOK ONLY
import warnings
warnings.filterwarnings('ignore')
import os, sys
sys.path.insert(0, os.path.abspath('../../utils')) # use directory to mmids.py

In [ ]:
# PYTHON 3
import numpy as np
from numpy import linalg as LA
from numpy.random import default_rng
rng = default_rng(535)
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
import mmids

In [ ]:
# FOR BOOK ONLY
#plt.rcParams['figure.figsize'] = [4.00, 2.25] # for high-def figs
#plt.rcParams['figure.dpi'] = 200 # for high-def figs

$\newcommand{\horz}{\rule[.5ex]{2.5ex}{0.5pt}}$

## Background: review of spectral decomposition

We review the spectral theorem.

### Eigenvalues and eigenvectors

We first review eigenvalues and eigenvectors. We work on $\mathbb{R}^d$.

**DEFINITION** **(Eigenvalues and Eigenvectors)** Let $A \in \mathbb{R}^{d \times d}$ be a square matrix. Then $\lambda \in \mathbb{R}$ is an eigenvalue of $A$ if there exists a nonzero vector $\mathbf{x} \neq \mathbf{0}$ such that

$$
A \mathbf{x} = \lambda \mathbf{x}.
$$

The vector $\mathbf{x}$ is referred to as an eigenvector. $\natural$

As the next example shows, not every matrix has a (real) eigenvalue.

**EXAMPLE:** **(No Real Eigenvalues)** Set $d = 2$ and let 

$$
A
=
\begin{pmatrix}
0 & -1 \\
1 & 0
\end{pmatrix}.
$$

For $\lambda$ to be an eigenvalue, there must be an nonzero eigenvector $\mathbf{x} = (x_1, x_2)^T$ such that

$$
A \mathbf{x} = \lambda \mathbf{x}
$$

or put differently

$$
- x_2 = \lambda x_1 
\quad\text{and}\quad
x_1 = \lambda x_2.
$$

Replacing these equations into each other, it must be that

$$
- x_2 = \lambda^2 x_2 
\quad\text{and}\quad
x_1 = - \lambda^2 x_1. 
$$

Because $x_1, x_2$ cannot both be $0$, $\lambda$ must satisfy the equation

$$
\lambda^2 = -1
$$

for which there is no real solution. $\lhd$

In general, $A \in \mathbb{R}^{d \times d}$ has at most $d$ distinct eigenvalues.

**LEMMA** **(Number of Eigenvalues)** Let $A \in \mathbb{R}^{d \times d}$ and 
let $\lambda_1, \ldots, \lambda_m$ be distinct eigenvalues of $A$ with corresponding
eigenvectors $\mathbf{x}_1, \ldots, \mathbf{x}_m$. Then $\mathbf{x}_1, \ldots, \mathbf{x}_m$ are linearly independent. In particular, $m \leq d$. $\flat$

**EXAMPLE:** **(Diagonal (and Similar) Matrices)** We use the notation $\mathrm{diag}(\lambda_1,\ldots,\lambda_d)$ for the diagonal matrix with diagonal entries $\lambda_1,\ldots,\lambda_d$. The upper bound in the *Number of Eigenvalues Lemma* can be achieved, for instance, by diagonal matrices with distinct diagonal entries $A = \mathrm{diag}(\lambda_1, \ldots, \lambda_d)$. Each standard basis vector $\mathbf{e}_i$ is then an eigenvector

$$
A \mathbf{e}_i = \lambda_i \mathbf{e}_i.
$$

More generally, let $A$ be similar to a matrix $D = \mathrm{diag}(\lambda_1, \ldots, \lambda_d)$ with distinct diagonal entries, that is, there exists a nonsingular matrix $P$ such that 

$$
A = P D P^{-1}.
$$

Let $\mathbf{p}_1, \ldots, \mathbf{p}_d$ be the columns of $P$. Note that, because the columns of $P$ form a basis of $\mathbb{R}^d$, the entries of the vector $\mathbf{c} = P^{-1} \mathbf{x}$ are the coefficients of the unique linear combination of the $\mathbf{p}_i$'s equal to $\mathbf{x}$. Indeed, $P \mathbf{c} = \mathbf{x}$. Hence, $A \mathbf{x}$ is can be thought of as: (1) expressing $\mathbf{x}$ in the basis $\mathbf{p}_1, \ldots, \mathbf{p}_d$, (2) and scaling the $\mathbf{p}_i$'s by the corresponding $\lambda_i$'s. In particular, the $\mathbf{p}_i$'s are eigenvectors of $A$ since, by the above, $P^{-1} \mathbf{p}_i = \mathbf{e}_i$ and so 

$$
A \mathbf{p}_i 
= P D P^{-1} \mathbf{p}_i 
= P D \mathbf{e}_i
= P (\lambda_i \mathbf{e}_i)
= \lambda_i \mathbf{p}_i.
$$

$\lhd$

**NUMERICAL CORNER:** In Python, the eigenvalues and eigenvectors of a matrix can be computed using [`numpy.linalg.eig`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.eig.html).

In [ ]:
A = np.array([[2.5, -0.5], [-0.5, 2.5]])

In [ ]:
w, v = LA.eig(A)
print(w)
print(v)

$\unlhd$

### Spectral theorem

When $A$ is symmetric, that is, $a_{ij} = a_{ji}$ for all $i,j$, a remarkable result is that there exists an orthonormal basis of $\mathbb{R}^d$ made of eigenvectors of $A$. Put differently, $A$ is similar to a diagonal matrix by an orthogonal transformation. We will prove this result, which you may have encountered in a previous course, in a section below. 

**THEOREM** **(Spectral Theorem)** Let $A \in \mathbb{R}^{d \times d}$ be a symmetric matrix, that is, $A^T = A$. Then $A$ has $d$ orthonormal eigenvectors $\mathbf{q}_1, \ldots, \mathbf{q}_d$ with corresponding (not necessarily distinct) real eigenvalues $\lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_d$. In matrix form, this is written as the matrix factorization

$$
A 
= Q \Lambda Q^T
= \sum_{i=1}^d \lambda_i \mathbf{q}_i \mathbf{q}_i^T
$$

where $Q$ has columns $\mathbf{q}_1, \ldots, \mathbf{q}_d$ and $\Lambda = \mathrm{diag}(\lambda_1, \ldots, \lambda_d)$. We refer to this factorization as a spectral decomposition of $A$.

$\sharp$

Note that this decomposition indeed produces the eigenvectors of $A$. For any $j$, we have

$$
A \mathbf{q}_j 
= \sum_{i=1}^d \lambda_i \mathbf{q}_i \mathbf{q}_i^T \mathbf{q}_j
= \lambda_j \mathbf{q}_j, 
$$

where we used that, by orthonormality, $\mathbf{q}_i^T \mathbf{q}_j = 0$ if $i \neq j$ and $\mathbf{q}_i^T \mathbf{q}_j = 1$ if $i = j$. The equation above says precisely that $\mathbf{q}_j$ is an eigenvector of $A$ with corresponding eigenvalue $\lambda_j$. Since we have found $d$ eigenvalues (possibly with repetition), we have found all of them.

**Remark:** It can be shown that the sequence of eigenvalues in the *Spectral Theorem* is unique, but that the eigenvectors are not. This will not play a role in our applications. 

### The case of positive semidefinite matrices


The eigenvalues of a symmetric matrix -- while real -- may be negative. There is however an important special case where the eigenvalues are nonnegative, positive semidefinite matrices.

**THEOREM** **(Characterization of Positive Semidefiniteness)** Let $A \in \mathbb{R}^{d \times d}$ be a symmetric matrix and let $A = Q \Lambda Q^T$ be a spectral decomposition of $A$ with $\Lambda = \mathrm{diag}(\lambda_1, \ldots, \lambda_d)$. Then $A \succeq 0$ if and only if its eigenvalues $\lambda_1, \ldots, \lambda_d$ are nonnegative. $\sharp$


*Proof:* Assume $A \succeq 0$. Let $\mathbf{q}_i$ be an eigenvector of $A$ with corresponding eigenvalue $\lambda_i$. Then

$$
\langle \mathbf{q}_i, A \mathbf{q}_i \rangle
= \langle \mathbf{q}_i, \lambda_i \mathbf{q}_i \rangle
= \lambda_i
$$

which must be nonnegative by definition of a positive semidefinite matrix. 

In the other direction, assume $\lambda_1, \ldots, \lambda_d \geq 0$. Note that the vector $Q^T \mathbf{x}$ has entries $\langle \mathbf{q}_i, \mathbf{x}\rangle$, $i=1,\ldots, d$. Thus

$$
\langle \mathbf{x}, A \mathbf{x} \rangle
= \mathbf{x}^T \left(\sum_{i=1}^d \lambda_i \mathbf{q}_i \mathbf{q}_i^T\right) \mathbf{x}
=  \sum_{i=1}^d \lambda_i \mathbf{x}^T\mathbf{q}_i \mathbf{q}_i^T\mathbf{x} 
= \sum_{i=1}^d \lambda_i \langle \mathbf{q}_i, \mathbf{x}\rangle^2
$$

which is necessarily nonnegative. $\square$

**EXAMPLE:** We have seen previously that the Laplacian matrix of a graph $G$ with $n$ vertices is positive semidefinite. Hence it has nonnegative eigenvalues, which are typically denoted

$$
0 \leq \mu_1 \leq \mu_2 \leq \cdots \leq \mu_n.
$$

$\lhd$